# Remove duplicates (BTXRD)
The original dataset is not changed. A copy is made in `data/BTXRD no duplicates`, and the duplicates are removed from the copy.

In [1]:
import hashlib
import json
import shutil
from pathlib import Path

import pandas as pd

BTXRD = Path("../data/BTXRD dataset")       # original, never changed
CLEAN = Path("../data/BTXRD no duplicates")  # copy, duplicates are removed here

MALIGNANT = {"osteosarcoma", "other mt"}  # all other tumor types are benign

if not CLEAN.exists():
    shutil.copytree(BTXRD, CLEAN)

# label and fingerprint (hash) of every image; exact copies have the same hash
rows = []
for path in sorted(BTXRD.glob("images/*.jp*g")):  # .jpg and .jpeg
    annotation = BTXRD / "Annotations" / (path.stem + ".json")
    if annotation.exists():
        tumor_type = json.loads(annotation.read_text())["shapes"][0]["label"]
        label = "malignant" if tumor_type in MALIGNANT else "benign"
    else:
        label = "normal"
    rows.append({"file": path.name, "label": label,
                 "hash": hashlib.md5(path.read_bytes()).hexdigest()})
btxrd = pd.DataFrame(rows)

# one row per duplicate pair
pairs = (btxrd[btxrd.duplicated("hash", keep=False)]
         .groupby("hash")
         .agg(image_1=("file", "first"), image_2=("file", "last"),
              label_1=("label", "first"), label_2=("label", "last"))
         .reset_index(drop=True))


def remove(file):
    """Delete an image and its annotation file (if it has one) from the copy."""
    (CLEAN / "images" / file).unlink(missing_ok=True)
    (CLEAN / "Annotations" / (Path(file).stem + ".json")).unlink(missing_ok=True)


print("Duplicate pairs:", len(pairs))

Duplicate pairs: 21


## Pairs with the same label

In [2]:
same_label = pairs[pairs["label_1"] == pairs["label_2"]]
print("Pairs with the same label:", len(same_label))
same_label

Pairs with the same label: 19


,image_1,image_2,label_1,label_2
0,IMG003669.jpeg,IMG003685.jpeg,normal,normal
1,IMG001952.jpeg,IMG002168.jpeg,normal,normal
2,IMG001733.jpeg,IMG001766.jpeg,benign,benign
3,IMG001873.jpeg,IMG003324.jpeg,normal,normal
4,IMG001721.jpeg,IMG001799.jpeg,benign,benign
5,IMG001948.jpeg,IMG002166.jpeg,normal,normal
6,IMG001953.jpeg,IMG002169.jpeg,normal,normal
7,IMG001734.jpeg,IMG001789.jpeg,benign,benign
8,IMG001951.jpeg,IMG002167.jpeg,normal,normal
9,IMG001673.jpeg,IMG001683.jpeg,malignant,malignant


In [3]:
# keep image_1, remove image_2
for file in same_label["image_2"]:
    remove(file)
print("Removed:", list(same_label["image_2"]))

Removed: ['IMG003685.jpeg', 'IMG002168.jpeg', 'IMG001766.jpeg', 'IMG003324.jpeg', 'IMG001799.jpeg', 'IMG002166.jpeg', 'IMG002169.jpeg', 'IMG001789.jpeg', 'IMG002167.jpeg', 'IMG001683.jpeg', 'IMG002173.jpeg', 'IMG000765.jpeg', 'IMG001836.jpeg', 'IMG002171.jpeg', 'IMG002172.jpeg', 'IMG003686.jpeg', 'IMG002170.jpeg', 'IMG000497.jpeg', 'IMG000683.jpeg']


## Pairs with different labels

In [4]:
different_label = pairs[pairs["label_1"] != pairs["label_2"]]
print("Pairs with different labels:", len(different_label))
different_label

Pairs with different labels: 2


,image_1,image_2,label_1,label_2
15,IMG000591.jpeg,IMG003456.jpeg,benign,normal
19,IMG000707.jpeg,IMG003346.jpeg,benign,normal


In [5]:
# the right label is unknown, so remove both images
for file in list(different_label["image_1"]) + list(different_label["image_2"]):
    remove(file)
print("Removed:", list(different_label["image_1"]) + list(different_label["image_2"]))
print("Images left:", len(list(CLEAN.glob("images/*"))))
print("Annotation files left:", len(list(CLEAN.glob("Annotations/*.json"))))

Removed: ['IMG000591.jpeg', 'IMG000707.jpeg', 'IMG003456.jpeg', 'IMG003346.jpeg']
Images left: 3723
Annotation files left: 1857
